LAB 8

In [23]:
con.execute("CREATE OR REPLACE TABLE silver_transactions AS SELECT * FROM read_parquet('bigdata/silver/transactions_enriched.parquet')")

# 1. Segmentação por score
print(con.execute("""
    SELECT segment, ROUND(AVG(credit_score), 1) AS score_medio, COUNT(DISTINCT customer_id) AS clientes
    FROM silver_transactions GROUP BY segment ORDER BY score_medio
""").fetchdf())

# 2. Risco por tipo de transação
print(con.execute("""
    SELECT transaction_type, SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraudes, COUNT(*) AS total,
           ROUND(100.0*SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END)/COUNT(*), 2) AS taxa_pct
    FROM silver_transactions GROUP BY transaction_type ORDER BY taxa_pct DESC
""").fetchdf())

# 3. Padrões temporais
print(con.execute("""
    SELECT day, COUNT(*) AS total_transacoes, SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraudes
    FROM silver_transactions GROUP BY day ORDER BY fraudes DESC LIMIT 10
""").fetchdf())

# 4. Cohort — clientes antigos vs novos
con.execute("CREATE OR REPLACE TABLE bronze_customers AS SELECT * FROM read_parquet('bigdata/bronze/customers.parquet')")
print(con.execute("""
    SELECT
        CASE WHEN DATEDIFF('day', c.created_at, CURRENT_DATE) > 365 THEN 'cliente antigo' ELSE 'cliente novo' END AS cohort,
        ROUND(AVG(t.amount), 2) AS ticket_medio, COUNT(*) AS transacoes
    FROM silver_transactions t JOIN bronze_customers c ON t.customer_id = c.customer_id
    GROUP BY 1
""").fetchdf())

# 5. Cross-sell — clientes Premium de alta frequência
print(con.execute("""
    SELECT customer_id, COUNT(*) AS compras FROM silver_transactions
    WHERE segment = 'Premium' AND transaction_type = 'compra'
    GROUP BY customer_id HAVING COUNT(*) > 10 ORDER BY compras DESC LIMIT 10
""").fetchdf())

     segment  score_medio  clientes
0   Standard        643.9      2665
1    Premium        652.7      5563
2  High-Risk        659.8       876
  transaction_type  fraudes  total  taxa_pct
0    transferencia    390.0  20102      1.94
1        pagamento    196.0  10180      1.93
2           compra    929.0  49782      1.87
3            saque    318.0  19936      1.60
   day  total_transacoes  fraudes
0   21              7777    161.0
1   26              7913    157.0
2   24              7792    148.0
3   27              7801    144.0
4   20              7850    140.0
5   25              7689    140.0
6   23              7696    133.0
7   22              7838    132.0
8   28              7914    131.0
9   11              1505     41.0
           cohort  ticket_medio  transacoes
0  cliente antigo        182.78       55070
1    cliente novo        185.03       44930
   customer_id  compras
0         1920       49
1         8500       44
2         7531       41
3         5251       40
4    